# Using Rocky Worlds DDT Utils to Discover JWST Observations of Exoplanets in Targets Under Consideration

The examples below use the `utils.py` file in the Rocky Worlds DDT Python API to discover observations of targets under consideration that might already exist. In this notebook we will use two targets under consideration: `LTT 1445 A b` and `TRAPPIST-1 g`.

* Import Rocky Worlds DDT utility functions
* Obtain exoplanet metadata from NASA NexSci Exoplanet Archive (ra, dec, period, planetary ephemeris).
* Obtain JWST Observations of the same region (potential transits, phase curves, secondary transits).
* Use JWST observation metadata of observation start and end times along with the targets period and planetary ephemeris to see if there are potential exoplanet event.
* Build an astropy table and add the orbit, phase and potential exoplanet event around the target

In [ ]:
from rocky_worlds_ddt.utils import check_jwst_observations, check_jwst_observation_type, query_nexsci_archive, query_mast_jwst_archive

In [ ]:
target_name = "LTT 1445 A b"

#### Query NexSci Archive for metadata we need for this exercise and print results.

In [ ]:
exoplanet_data, preferred_idx = query_nexsci_archive(target_name)
exoplanet_data

#### The preferred index variable (`preferred_idx`) is the value of the row from the table where the NexSci table column for `default_flag==1`. This are the preferred values for system, but others are available. 

In [ ]:
print(preferred_idx)

#### Let's take a look at the values returned from our API queries.

* Here we strip the units (degrees) from ra and dec. This is because the query of the MAST Archive (following cells) only accept python float types. If we pass the units still wrapped in the array (`exoplanet_data[preferred_idx]["ra"]`) the query will fail.
* For the values of `pl_tranmid` and `pl_orbper`, we keep the units (`d` which is short for `days`).

In [ ]:
ra = exoplanet_data[preferred_idx]["ra"].value[0]
dec = exoplanet_data[preferred_idx]["dec"].value[0]
T0 = exoplanet_data[preferred_idx]["pl_tranmid"][0]
P = exoplanet_data[preferred_idx]["pl_orbper"][0]
print(ra, dec, T0, P)

#### Now that we have the metadata for the exoplanet system we are interested in, let's see if there are any JWST observations in the same region.

In [ ]:
jwst_observations = query_mast_jwst_archive(ra, dec)
jwst_observations

#### For the host `LTT 1445A` there are many observations, but for the exoplanet (`LTT 1445A b`), are there any events in the time period of the JWST observations listed above? We will use the columns `date_obs` and `duration` coupled with `pl_tranmid` and `pl_orbper` to calculate phase of the exoplanet which we will use to see if there is an event occuring during the time of the JWST observations in this table.

In [ ]:
observation_types = check_jwst_observation_type(target_name, period=P, planet_ephemeris=T0, jwst_observations=jwst_observations)
observation_types

#### For the Rocky Worlds DDT, we are interest in secondary eclipses, let's see if there is the possiblity for that type of event for our target in the observations returned from our MAST query.

In [ ]:
observation_types[observation_types["obs_type"]=="SECONDARY ECLIPSE"]

#### According to our algorithm, there are three observations with potential secondary eclipses.

### Now we are going to look at another Target Under Consideration, `TRAPPIST-1 g`. This target has issues with data in the NexSci Database, the preferred database entry is missing an important piece of data to calculate the phase.

**Note:** This notebook was written in January 2025, this issue could disappear if preferred data entry in the NexSci database is changed or if new measurements are made. This example is a work around to obtain the information we are interested in.

In [ ]:
failed_target_name = "TRAPPIST-1 g"

In [ ]:
failed_exoplanet_data, failed_preferred_idx = query_nexsci_archive(failed_target_name)
failed_exoplanet_data

#### When we look at the metadata from the preferred row, the planetary ephemeris (`T0` or `pl_tranmid`) is NaN.

In [ ]:
failed_ra = failed_exoplanet_data[failed_preferred_idx]["ra"].value[0]
failed_dec = failed_exoplanet_data[failed_preferred_idx]["dec"].value[0]
failed_T0 = failed_exoplanet_data[failed_preferred_idx]["pl_tranmid"][0]
failed_P = failed_exoplanet_data[failed_preferred_idx]["pl_orbper"][0]
print(failed_ra, failed_dec, failed_T0, failed_P)

#### In this case, we can use the planet ephemeris from Gillon et al 2017.

In [ ]:
successful_T0 = failed_exoplanet_data[[2]]["pl_tranmid"][0]
successful_reference = failed_exoplanet_data[[2]]["pl_refname"][0]
print(successful_T0)
print(successful_reference)

In [ ]:
failed_jwst_observations = query_mast_jwst_archive(failed_ra, failed_dec)
failed_jwst_observations

In [ ]:
trappist_1_g_observation_types = check_jwst_observation_type(failed_target_name, period=failed_P, planet_ephemeris=successful_T0, jwst_observations=failed_jwst_observations)
trappist_1_g_observation_types

#### `TRAPPIST-1 g` has one potential secondary eclipse. 

In [ ]:
trappist_1_g_observation_types[trappist_1_g_observation_types["obs_type"]=="SECONDARY ECLIPSE"]

## Programmatically Build List of All Events for targets in Targets Under Consideration List

In [ ]:
from pathlib import Path
from astropy.io import ascii
from astropy.table import vstack
import numpy as np

In [ ]:
tuc_file = Path("../data/tuc.txt")
tuc_data = ascii.read(tuc_file, format="fixed_width", header_rows=["name", "dtype"])
tuc_data["planet_name"]

In [ ]:
all_observation_type_tables = []

for planet_name in tuc_data["planet_name"]:
    # Ignore Solar System Planets
    if planet_name in ["Earth", "Mars", "Venus", "Mercury"]:
        continue
    else:
        # Get exoplanet data from NexSci
        planet_data, _ = query_nexsci_archive(planet_name)
        # Get same data, but ensure planet ephemeris and period are NOT value NaN
        nan_free_planet_data = planet_data[~np.isnan(planet_data["pl_tranmid"]) & ~np.isnan(planet_data["pl_orbper"])]

        if len(nan_free_planet_data) == 0:
            print(f"No complete datasets for {planet_name}")
            continue

        # Check to see if preferred index is still in this updated table
        preferred_index = np.where(nan_free_planet_data["default_flag"]==1)[0]
        if preferred_index.size > 0:
            T0 = nan_free_planet_data[preferred_index]["pl_tranmid"][0]
            P = nan_free_planet_data[preferred_index]["pl_orbper"][0]
            ra = nan_free_planet_data[preferred_index]["ra"].value[0]
            dec = nan_free_planet_data[preferred_index]["dec"].value[0]
        else:
            # Just use first entry with complete data
            T0 = nan_free_planet_data[0]["pl_tranmid"]
            P = nan_free_planet_data[0]["pl_orbper"]
            ra = nan_free_planet_data[0]["ra"].value
            dec = nan_free_planet_data[0]["dec"].value

        jwst_observations = query_mast_jwst_archive(ra, dec)
        if len(jwst_observations) == 0:
            print(f"No JWST observations for {planet_name}")
            continue
        else:
            print(f"Found {len(jwst_observations)} potential {planet_name} observations")

        observation_type_table = check_jwst_observation_type(planet_name, period=P, planet_ephemeris=T0, jwst_observations=jwst_observations)
        all_observation_type_tables.append(observation_type_table)

combine_all_observation_tables = vstack(all_tables)

In [ ]:
combine_all_observation_tables

In [ ]:
secondary_eclipses = np.where((combine_all_observation_tables["obs_type"] == "SECONDARY ECLIPSE")
                             & (combine_all_observation_tables["access"] == "PUBLIC"))
all_public_secondary_eclipses = combine_all_observation_tables[secondary_eclipses]["planet_name", "ArchiveFileID", "instrume", "exp_type", "opticalElements", "date_obs", "program", "access"]

In [ ]:
ascii.write(
    combine_all_observation_tables,
    "../data/all_tuc_events.txt",
    format="fixed_width",
    header_rows=["name", "dtype"],
    overwrite=True,
)

ascii.write(
        all_public_secondary_eclipses,
        "../data/all_public_tuc_secondary_eclipses.txt",
        format="fixed_width",
        header_rows=["name", "dtype"],
        overwrite=True,
    )